In [98]:
import pandas as pd

# Sample DataFrames
df_invoice = pd.read_csv("freight_table.csv")        # Includes supplier_name, any_priority_commodity
df_mode = pd.read_csv("supplier_mode.csv")                 # Includes supplier_name, supplier_mode

# Step 1: Merge supplier mode
df = df_invoice.merge(df_mode, on="supplier_name", how="left")

# Step 2: Derive supplier_mode_type
def classify_supplier(row):
    if row['supplier_mode'] == "Manufacturer":
        if row['any_invoice_priority_products_2008']:
            return "Manufacturer Priority"
        else:
            return "Manufacturer Non Priority"
    elif row['supplier_mode'] == "Distributor":
        return "Distributor"
    else:
        return "Unknown"


c:\Users\nzhuw\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (2,12,44) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [99]:
df.head(2)

,site,site_description,supplier_no,supplier_name,invoice_id,invoice_no,date_posted,project_id,project_name,account,...,any_invoice_priority_products_2008,freight_per_invoice,priority_product_total,non_priority_product_total,percentage_priority,percentage_non_priority,pct_priority_greater_than_70,invoice_total,supplier_mode,supplier_mode_real
0,SPN,Spectra Norcross,103277,William M. Bird,531030,656881,45352,2401132763,FAROPOINT LOBBY RENOVATION,2008,...,True,5.79,1343.25,0.0,100.0,0.0,True,1349.04,Distributor,Distributor
1,SPN,Spectra Norcross,103423,"Shaw Industries, Inc.",561080,9550085,45371,2401132815,PCI-SCU Griffin Carpet Replacement,2008,...,True,2.02,1749.32,0.0,100.0,0.0,True,1751.34,Manufacturer,Manufacturer


In [100]:

df["supplier_mode_type"] = df.apply(classify_supplier, axis=1)


In [101]:
df.columns

Index(['site', 'site_description', 'supplier_no', 'supplier_name',
       'invoice_id', 'invoice_no', 'date_posted', 'project_id', 'project_name',
       'account', 'account_description', 'planned_delivery_date',
       'ship_to_zip', 'po_no', 'po_line_no', 'po_rel_no', 'receipt_no',
       'part_no', 'part_description', 'comm_1', 'comm_2', 'po_purch_qty',
       'purch_uom', 'po_inv_qty', 'inv_uom', 'invoiced_line_qty',
       'invoice_line_total', 'po_price', 'supplier_name_mapped',
       'match_supplier', 'commodity_group_mapped', 'commodity_description',
       'commodity_code', 'priority_commodity', 'match_commodity',
       'is_classified', 'line_classification', 'new_commodity_description',
       'new_commodity_group', 'conversion_code', 'has_freight_line',
       'multiple_freight_lines', 'multiple_parts', 'multiple_commodities',
       'priority_multiple_commodities', 'all_invoice_priority_products_2008',
       'any_invoice_priority_products_2008', 'freight_per_invoice',
  

In [102]:
df = df[df['site'].isin(['SPJ','SPT','SPW','SPN'])]

In [103]:
pivot = df.pivot_table(
    index=['site','any_invoice_priority_products_2008'],
    columns="supplier_mode_type",
    values="freight_per_invoice",
    aggfunc="sum"
)
pivot = pivot.reset_index()
pivot

supplier_mode_type,site,any_invoice_priority_products_2008,Distributor,Manufacturer Non Priority,Manufacturer Priority,Unknown
0,SPJ,False,86258.50,71144.73,NaN,150.00
1,SPJ,True,19625.62,NaN,141510.73,2199.24
2,SPN,False,412674.50,429740.08,NaN,35181.64
3,SPN,True,49887.79,NaN,201762.12,1293.94
4,SPT,False,207610.92,297382.28,NaN,10443.15
5,SPT,True,57042.88,NaN,360791.68,1700.00
6,SPW,False,308771.97,303608.51,NaN,10289.30
7,SPW,True,73210.72,NaN,443965.10,5648.23


In [104]:
def summarize_supplier_costs_by_site(pivot_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize freight costs per site:
    - Adds Distributor + Unknown together
    - Gets Manufacturer Priority from True row per site
    - Gets Manufacturer Non Priority from False row per site

    Args:
        pivot_df (pd.DataFrame): Pivoted DataFrame with:
            ['site', 'any_invoice_priority_products_2008'] as columns
            and supplier_mode_type columns like 'Distributor', 'Unknown', etc.

    Returns:
        pd.DataFrame: Summary table with one row per site.
    """
    # Ensure columns are accessible
    if 'site' not in pivot_df.columns or 'any_invoice_priority_products_2008' not in pivot_df.columns:
        raise ValueError("Expected columns: ['site', 'any_invoice_priority_products_2008']")

    # Initialize result rows
    summary_rows = []

    # Get unique sites
    for site in pivot_df['site'].unique():
        site_df = pivot_df[pivot_df['site'] == site]

        distributor_total = 0
        mp = 0
        mnp = 0

        for _, row in site_df.iterrows():
            is_priority = row['any_invoice_priority_products_2008']
            distributor = row.get('Distributor', 0) or 0
            unknown = row.get('Unknown', 0) or 0

            distributor_total += distributor + unknown

            if is_priority is True:
                mp = row.get('Manufacturer Priority', 0) or 0
            elif is_priority is False:
                mnp = row.get('Manufacturer Non Priority', 0) or 0

        summary_rows.append({
            'site': site,
            'Manufacturer(Priority)': mp,
            'Manufacturer(NonPriority)': mnp,
            'Distributor': distributor_total
        })

    return pd.DataFrame(summary_rows)


In [105]:
def add_escalated_rows_with_measure_column(summary_df: pd.DataFrame, escalation_table: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a new row per site applying the escalation factor to Manufacturer costs only.
    Labels rows using a 'measure' column instead of modifying the site name.

    Args:
        summary_df (pd.DataFrame): Original summary table with one row per site.
        escalation_table (pd.DataFrame): Lookup table with ['site', 'escalation_factor'].

    Returns:
        pd.DataFrame: Expanded table with 'measure' column ('Base' and 'Escalated').
    """
    # Merge escalation factor into summary
    merged = summary_df.merge(escalation_table, on='site', how='left')

    if merged['escalation_factor'].isnull().any():
        missing = merged[merged['escalation_factor'].isnull()]['site'].tolist()
        raise ValueError(f"Missing escalation_factor for site(s): {missing}")

    rows = []

    for _, row in merged.iterrows():
        # Base row
        rows.append({
            'site': row['site'],
            'measure': 'Base',
            'Manufacturer(Priority)': row['Manufacturer(Priority)'],
            'Manufacturer(NonPriority)': row['Manufacturer(NonPriority)'],
            'Distributor': row['Distributor']
        })

        # Escalated row
        rows.append({
            'site': row['site'],
            'measure': 'Escalated',
            'Manufacturer(Priority)': row['Manufacturer(Priority)'] * (1-row['escalation_factor']),
            'Manufacturer(NonPriority)': row['Manufacturer(NonPriority)'] * (1-row['escalation_factor']),
            'Distributor': row['Distributor']  # unchanged
        })

    return pd.DataFrame(rows)


In [106]:
final_df = summarize_supplier_costs_by_site(pivot)
print(final_df)



  site  Manufacturer(Priority)  Manufacturer(NonPriority)  Distributor
0  SPJ               141510.73                   71144.73    108233.36
1  SPN               201762.12                  429740.08    499037.87
2  SPT               360791.68                  297382.28    276796.95
3  SPW               443965.10                  303608.51    397920.22


In [107]:
summary = summarize_supplier_costs_by_site(pivot)

# Example escalation table
site_escalation = pd.DataFrame({
    'site': ['SPJ','SPT','SPW','SPN'],
    'escalation_factor': [0.07,0.1,0.24,0.15]
})

summary_with_escalation = add_escalated_rows_with_measure_column(summary, site_escalation)
summary_with_escalation


,site,measure,Manufacturer(Priority),Manufacturer(NonPriority),Distributor
0,SPJ,Base,141510.7300,71144.7300,108233.36
1,SPJ,Escalated,131604.9789,66164.5989,108233.36
2,SPN,Base,201762.1200,429740.0800,499037.87
3,SPN,Escalated,171497.8020,365279.0680,499037.87
4,SPT,Base,360791.6800,297382.2800,276796.95
5,SPT,Escalated,324712.5120,267644.0520,276796.95
6,SPW,Base,443965.1000,303608.5100,397920.22
7,SPW,Escalated,337413.4760,230742.4676,397920.22


In [108]:
summary_with_escalation.to_excel("21062025_all_site_invoice_freight_v1300.xlsx", index=False)